# 100 — Interactive Betsy wiggle picker

RefraPy is not reading geometry from our SEG-Y headers reliably, so this notebook provides a direct Python fallback for review and picking.

Features:
- loads the Betsy nodal and Geode gathers directly from the catalog;
- uses real receiver positions from `trace_index` / Geode metadata;
- plots normalized wiggles by receiver position, not trace index;
- controls gain, clip, time window, shading mode, and dataset;
- supports click picking on the gather;
- writes picks to CSV and SQLite.

Inputs:
- `shot_events`, `shot_gather_files`, `trace_index`, `geode_events`

Known Betsy IDs:
- nodal: `T1_N2_Refraction2m_T1_N2_E00449`
- Geode: `GEODE_T1_2M_REFRACTION_F3088`

## 1. Configuration

In [1]:
from pathlib import Path
import sqlite3
import json
import traceback

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from obspy import read, Stream, UTCDateTime
from scipy.signal import correlate, correlation_lags

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    HAS_WIDGETS = True
except Exception:
    HAS_WIDGETS = False

PROJECT_ROOT = Path("/Volumes/tachyon/LBSSP_DATA")
CATALOG_DB = PROJECT_ROOT / "catalog" / "lbssp_shot_catalog.sqlite"

OUT_ROOT = PROJECT_ROOT / "betsy_gun_alignment_v1" / "interactive_picker"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

BETSY_NODAL_EVENT_ID = "T1_N2_Refraction2m_T1_N2_E00449"
BETSY_GEODE_EVENT_ID = "GEODE_T1_2M_REFRACTION_F3088"

NODAL_COMPONENTS = ["Z", "N", "E"]
GEODE_COMPONENT = None   # Geode channels may not be named *Z.

DEFAULT_TMIN = 0.0
DEFAULT_TMAX = 0.8

print("CATALOG_DB:", CATALOG_DB)
print("OUT_ROOT:", OUT_ROOT)
print("ipywidgets:", HAS_WIDGETS)

CATALOG_DB: /Volumes/tachyon/LBSSP_DATA/catalog/lbssp_shot_catalog.sqlite
OUT_ROOT: /Volumes/tachyon/LBSSP_DATA/betsy_gun_alignment_v1/interactive_picker
ipywidgets: True


## 2. Load catalog rows and waveforms

In [2]:
conn = sqlite3.connect(CATALOG_DB)

shot_events = pd.read_sql("SELECT * FROM shot_events", conn)
shot_gather_files = pd.read_sql("SELECT * FROM shot_gather_files", conn)
trace_index = pd.read_sql("SELECT * FROM trace_index WHERE instrument_system='nodal'", conn)
geode_events = pd.read_sql("SELECT * FROM geode_events", conn)

nodal_event = shot_events[shot_events["event_id"].astype(str).eq(BETSY_NODAL_EVENT_ID)].iloc[0]
geode_event = geode_events[geode_events["geode_event_id"].astype(str).eq(BETSY_GEODE_EVENT_ID)].iloc[0]

nodal_mseed = shot_gather_files[
    (shot_gather_files["event_id"].astype(str).eq(BETSY_NODAL_EVENT_ID)) &
    (shot_gather_files["file_type"].astype(str).str.lower().eq("mseed"))
]["file_path"].iloc[0]

geode_path = geode_event["geode_file_path"]
source_x_m = float(geode_event["source_x_m"])

print("Nodal:", BETSY_NODAL_EVENT_ID)
print(" ", nodal_mseed, Path(nodal_mseed).exists())
print("Geode:", BETSY_GEODE_EVENT_ID)
print(" ", geode_path, Path(geode_path).exists())
print("Source x:", source_x_m)
display(pd.DataFrame([nodal_event]).T)
display(pd.DataFrame([geode_event]).T)

Nodal: T1_N2_Refraction2m_T1_N2_E00449
  /Volumes/tachyon/LBSSP_DATA/nodal_fullnode_shotgathers_v4/T1_N2_Refraction2m/gathers_mseed/T1_N2_Refraction2m_T1_N2_E00449_DPall.mseed True
Geode: GEODE_T1_2M_REFRACTION_F3088
  /Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_051826/3088.dat True
Source x: 47.0


,2787
event_id,T1_N2_Refraction2m_T1_N2_E00449
instrument_system,nodal
line,T1
transect,None
survey,T1_2m_refraction
survey_type,geode_stack_metadata
shot_no,42.0
file_no,3088.0
source_x_m,47.0
source_type,hammer


,87
geode_event_id,GEODE_T1_2M_REFRACTION_F3088
instrument_system,geode
source_sheet,T1_2m_Refraction
line,T1
survey,T1_2m_refraction
survey_type,refraction
nodal_timewindow_label,T1_N2_Refraction2m
shot_no,42.0
file_no,3088.0
source_x_m,47.0


## 3. Geometry and gather helpers

In [3]:
def select_component(st, component):
    if component is None:
        return st.copy()
    stc = st.select(channel=f"*{component}").copy()
    return stc


def shift_stream_to_relative_time(st):
    if len(st) == 0:
        return st
    origin = min(tr.stats.starttime for tr in st)
    out = st.copy()
    for tr in out:
        rel = tr.stats.starttime - origin
        tr.stats.starttime = UTCDateTime(0) + rel
    return out


def attach_nodal_geometry(st, event_id, component=None, source_x_m=None):
    geom = trace_index[trace_index["event_id"].astype(str).eq(str(event_id))].copy()
    if component is not None:
        geom = geom[geom["channel"].astype(str).str.endswith(component)].copy()

    geom["receiver_x_m"] = pd.to_numeric(geom["receiver_x_m"], errors="coerce")
    geom = geom.dropna(subset=["receiver_x_m"])

    lookup = {}
    for _, r in geom.iterrows():
        station = str(r.get("station", ""))
        channel = str(r.get("channel", ""))
        network = str(r.get("network", ""))
        location = "" if pd.isna(r.get("location", "")) else str(r.get("location", ""))
        lookup[(station, channel)] = float(r["receiver_x_m"])
        lookup[f"{network}.{station}.{location}.{channel}"] = float(r["receiver_x_m"])
        if "seed_id" in geom.columns and pd.notna(r.get("seed_id")):
            lookup[str(r["seed_id"])] = float(r["seed_id"]) if False else float(r["receiver_x_m"])

    out = st.copy()
    n = 0
    for tr in out:
        x = lookup.get((str(tr.stats.station), str(tr.stats.channel)))
        if x is None:
            x = lookup.get(tr.id)
        if x is None:
            try:
                x = float(str(tr.stats.station)) / 100.0
            except Exception:
                x = None
        if x is not None and np.isfinite(float(x)):
            tr.stats.receiver_x_m = float(x)
            n += 1
        if source_x_m is not None:
            tr.stats.source_x_m = float(source_x_m)
    return out, n


def infer_geode_receiver_xs(geode_row, n_traces):
    first = pd.to_numeric(geode_row.get("receiver_first_m"), errors="coerce")
    last = pd.to_numeric(geode_row.get("receiver_last_m"), errors="coerce")
    spacing = pd.to_numeric(geode_row.get("receiver_spacing_m"), errors="coerce")

    if np.isfinite(first) and np.isfinite(spacing) and spacing != 0:
        return first + np.arange(n_traces) * spacing, "receiver_first_m + spacing"
    if np.isfinite(first) and np.isfinite(last) and n_traces > 1:
        return np.linspace(first, last, n_traces), "linspace(first,last)"
    return np.arange(n_traces, dtype=float), "trace_index_fallback"


def attach_geode_geometry(st, geode_row):
    out = st.copy()
    xs, method = infer_geode_receiver_xs(geode_row, len(out))
    sx = float(geode_row["source_x_m"])
    for tr, x in zip(out, xs):
        tr.stats.receiver_x_m = float(x)
        tr.stats.source_x_m = sx
    return out, method


def preprocess(st, freqmin=5.0, freqmax=150.0):
    out = st.copy()
    for tr in out:
        tr.data = tr.data.astype(np.float64)
        tr.detrend("linear")
        tr.taper(max_percentage=0.02)
        nyq = 0.5 / tr.stats.delta
        if freqmax and freqmax < 0.95 * nyq:
            tr.filter("bandpass", freqmin=freqmin, freqmax=freqmax, corners=4, zerophase=True)
        else:
            tr.filter("highpass", freq=freqmin, corners=4, zerophase=True)
    return out


def stream_to_arrays(st, tmin=0.0, tmax=0.8):
    rows = []
    for i, tr in enumerate(st):
        x = float(getattr(tr.stats, "receiver_x_m", i))
        dt = float(tr.stats.delta)
        t0 = tr.stats.starttime - UTCDateTime(0)
        t = t0 + np.arange(tr.stats.npts) * dt
        grid = np.arange(tmin, tmax + 0.5 * dt, dt)
        y = np.interp(grid, t, tr.data.astype(float), left=np.nan, right=np.nan)
        if np.isfinite(y).sum() < 5:
            continue
        rows.append({"x": x, "t": grid, "y": y, "id": tr.id, "station": tr.stats.station, "channel": tr.stats.channel})
    return sorted(rows, key=lambda r: r["x"])


def normalize_trace(y, clip_percentile=99.0):
    y = np.asarray(y, dtype=float)
    y = y - np.nanmedian(y)
    s = np.nanpercentile(np.abs(y), clip_percentile)
    if not np.isfinite(s) or s <= 0:
        s = np.nanmax(np.abs(y))
    if not np.isfinite(s) or s <= 0:
        s = 1.0
    return np.clip(y / s, -1, 1)

## 4. Load and prepare Betsy streams

In [4]:
raw_nodal = read(str(nodal_mseed))
raw_geode = read(str(geode_path))

nodal_streams = {}
for comp in NODAL_COMPONENTS:
    stc = select_component(raw_nodal, comp)
    stc, n_attached = attach_nodal_geometry(stc, BETSY_NODAL_EVENT_ID, component=comp, source_x_m=source_x_m)
    stc = shift_stream_to_relative_time(stc)
    stc = preprocess(stc)
    nodal_streams[comp] = stc
    print(f"Nodal {comp}: {len(stc)} traces, geometry attached {n_attached}")

geode_stream = select_component(raw_geode, GEODE_COMPONENT)
geode_stream, geode_geom_method = attach_geode_geometry(geode_stream, geode_event)
geode_stream = shift_stream_to_relative_time(geode_stream)
geode_stream = preprocess(geode_stream)
print("Geode:", len(geode_stream), geode_geom_method)
print("Geode channels:", sorted(set(tr.stats.channel for tr in geode_stream)))

Nodal Z: 34 traces, geometry attached 34


/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/io/seg2/seg2.py:365: UserWarning: Many companies use custom defined SEG2 header variables. This might cause basic header information reflected in the single traces' stats to be wrong (e.g. recording delays, first sample number, station code names, ..). Please check the complete list of additional unmapped header fields that gets stored in Trace.stats.seg2 and/or the manual of the source of the SEG2 files for fields that might influence e.g. trace start times.
  warnings.warn(WARNING_HEADER)


Nodal N: 34 traces, geometry attached 34
Nodal E: 34 traces, geometry attached 34
Geode: 72 receiver_first_m + spacing
Geode channels: ['']


## 5. Interactive wiggle plot with click picking

In [5]:
PICKS = []

def plot_wiggle_arrays(
    rows,
    *,
    title,
    source_x_m=None,
    gain=1.0,
    trace_scale=0.75,
    clip_percentile=99.0,
    shade="positive",
    linewidth=0.6,
    color="black",
    picks=None,
    figsize=(13, 7),
):
    fig, ax = plt.subplots(figsize=figsize)

    if not rows:
        ax.text(0.5, 0.5, "No traces", transform=ax.transAxes, ha="center")
        return fig, ax

    xs = np.array([r["x"] for r in rows], dtype=float)
    ux = np.sort(np.unique(xs[np.isfinite(xs)]))
    dx = np.nanmedian(np.diff(ux)) if len(ux) > 1 else 1.0
    if not np.isfinite(dx) or dx <= 0:
        dx = 1.0

    for r in rows:
        x = r["x"]
        t = r["t"]
        y = normalize_trace(r["y"], clip_percentile=clip_percentile)
        wig = x + gain * trace_scale * dx * y

        ax.plot(wig, t, color=color, linewidth=linewidth)

        if shade in ["positive", "both"]:
            ax.fill_betweenx(t, x, wig, where=(wig >= x), color="red", alpha=0.35, interpolate=True)
        if shade in ["negative", "both"]:
            ax.fill_betweenx(t, x, wig, where=(wig < x), color="blue", alpha=0.25, interpolate=True)

    if source_x_m is not None:
        ax.axvline(source_x_m, linestyle="--", color="tab:green", linewidth=1.2, label=f"source x={source_x_m:.1f} m")

    if picks:
        for p in picks:
            ax.plot(p["receiver_x_m"], p["pick_time_s"], "o", markersize=5, color="tab:orange")
            ax.text(p["receiver_x_m"], p["pick_time_s"], f"{p['pick_time_s']:.3f}", fontsize=7)

    ax.invert_yaxis()
    ax.set_xlabel("Receiver x (m)")
    ax.set_ylabel("Time (s)")
    ax.set_title(title)
    ax.grid(True, alpha=0.25)
    if source_x_m is not None:
        ax.legend(loc="best")

    return fig, ax


def nearest_receiver(rows, x):
    xs = np.array([r["x"] for r in rows], dtype=float)
    i = int(np.nanargmin(np.abs(xs - x)))
    return rows[i], i


def make_picker(dataset="nodal_Z", tmin=0.0, tmax=0.8, gain=1.0, trace_scale=0.75, clip=99.0, shade="positive"):
    if dataset.startswith("nodal_"):
        comp = dataset.split("_", 1)[1]
        st = nodal_streams[comp]
        rows = stream_to_arrays(st, tmin=tmin, tmax=tmax)
    elif dataset == "geode":
        rows = stream_to_arrays(geode_stream, tmin=tmin, tmax=tmax)
    else:
        raise ValueError(dataset)

    title = f"Betsy Gun {dataset} | click to add pick; shift-click/right-click nearest existing pick to delete"

    fig, ax = plot_wiggle_arrays(
        rows,
        title=title,
        source_x_m=source_x_m,
        gain=gain,
        trace_scale=trace_scale,
        clip_percentile=clip,
        shade=shade,
        picks=[p for p in PICKS if p["dataset"] == dataset],
    )

    def onclick(event):
        if event.inaxes != ax or event.xdata is None or event.ydata is None:
            return

        r, idx = nearest_receiver(rows, event.xdata)
        x_pick = float(r["x"])
        t_pick = float(event.ydata)

        # right-click or shift-click deletes nearest pick for this dataset/receiver
        if event.button == 3 or getattr(event, "key", None) == "shift":
            subset = [
                (i, p) for i, p in enumerate(PICKS)
                if p["dataset"] == dataset and abs(p["receiver_x_m"] - x_pick) < 0.51
            ]
            if subset:
                # remove nearest in time
                j, _ = min(subset, key=lambda ip: abs(ip[1]["pick_time_s"] - t_pick))
                PICKS.pop(j)
                print(f"Deleted pick near x={x_pick:.2f}, t={t_pick:.3f}")
        else:
            PICKS.append({
                "dataset": dataset,
                "receiver_x_m": x_pick,
                "pick_time_s": t_pick,
                "trace_index": idx,
                "trace_id": r["id"],
                "station": r["station"],
                "channel": r["channel"],
                "source_x_m": source_x_m,
                "event_id": BETSY_NODAL_EVENT_ID if dataset.startswith("nodal") else BETSY_GEODE_EVENT_ID,
            })
            print(f"Added pick: {dataset} x={x_pick:.2f} t={t_pick:.4f}")

        plt.close(fig)
        make_picker(dataset, tmin, tmax, gain, trace_scale, clip, shade)

    cid = fig.canvas.mpl_connect("button_press_event", onclick)
    plt.show()
    return fig, ax


if HAS_WIDGETS:
    dataset_widget = widgets.Dropdown(
        options=["nodal_Z", "nodal_N", "nodal_E", "geode"],
        value="nodal_Z",
        description="Dataset",
    )
    tmin_widget = widgets.FloatText(value=DEFAULT_TMIN, description="tmin")
    tmax_widget = widgets.FloatText(value=DEFAULT_TMAX, description="tmax")
    gain_widget = widgets.FloatSlider(value=1.0, min=0.1, max=10.0, step=0.1, description="gain")
    scale_widget = widgets.FloatSlider(value=0.75, min=0.1, max=3.0, step=0.05, description="trace scale")
    clip_widget = widgets.FloatSlider(value=99.0, min=90.0, max=100.0, step=0.5, description="clip %")
    shade_widget = widgets.Dropdown(options=["none", "positive", "negative", "both"], value="positive", description="shade")

    ui = widgets.VBox([
        widgets.HBox([dataset_widget, shade_widget]),
        widgets.HBox([tmin_widget, tmax_widget]),
        widgets.HBox([gain_widget, scale_widget, clip_widget]),
    ])

    out = widgets.interactive_output(
        make_picker,
        {
            "dataset": dataset_widget,
            "tmin": tmin_widget,
            "tmax": tmax_widget,
            "gain": gain_widget,
            "trace_scale": scale_widget,
            "clip": clip_widget,
            "shade": shade_widget,
        },
    )

    display(ui, out)
else:
    print("ipywidgets not available. Use make_picker(dataset='nodal_Z', gain=2, shade='both') manually.")
    make_picker(dataset="nodal_Z", gain=1.0, shade="positive")

Output()

## 6. Static combined overlay plot

In [8]:
def plot_overlay_nodal_geode(comp="Z", geode_shift_s=0.0, tmin=0.0, tmax=0.8, gain=1.0, shade="positive"):
    nodal_rows = stream_to_arrays(nodal_streams[comp], tmin=tmin, tmax=tmax)
    geode_rows = stream_to_arrays(geode_stream, tmin=tmin - geode_shift_s, tmax=tmax - geode_shift_s)

    # Apply time shift to Geode rows for plotting.
    shifted_geode_rows = []
    for r in geode_rows:
        rr = dict(r)
        rr["t"] = rr["t"] + geode_shift_s
        shifted_geode_rows.append(rr)

    fig, ax = plt.subplots(figsize=(14, 8))

    plot_wiggle_arrays(
        nodal_rows,
        title=f"Betsy Gun overlay: nodal {comp} black, Geode red | Geode shift {geode_shift_s:+.4f} s",
        source_x_m=source_x_m,
        gain=gain,
        shade="none",
        color="black",
        picks=None,
        figsize=(14, 8),
    )
    plt.close()

    # Redo on shared axis.
    fig, ax = plt.subplots(figsize=(14, 8))
    for rows, color, alpha, label in [
        (nodal_rows, "black", 0.9, f"nodal {comp}"),
        (shifted_geode_rows, "tab:red", 0.55, "Geode shifted"),
    ]:
        xs = np.array([r["x"] for r in rows], dtype=float)
        ux = np.sort(np.unique(xs[np.isfinite(xs)]))
        dx = np.nanmedian(np.diff(ux)) if len(ux) > 1 else 1.0
        if not np.isfinite(dx) or dx <= 0:
            dx = 1.0
        for r in rows:
            y = normalize_trace(r["y"])
            wig = r["x"] + gain * 0.75 * dx * y
            ax.plot(wig, r["t"], color=color, alpha=alpha, linewidth=0.6)

    ax.axvline(source_x_m, linestyle="--", color="tab:green", linewidth=1.2)
    ax.invert_yaxis()
    ax.set_xlabel("Receiver x (m)")
    ax.set_ylabel("Time (s)")
    ax.set_title(f"Betsy Gun overlay: nodal {comp} + Geode shifted {geode_shift_s:+.4f} s")
    ax.grid(True, alpha=0.25)
    plt.show()
    return fig, ax

# Example:
# plot_overlay_nodal_geode(comp="Z", geode_shift_s=0.0, gain=2.0)

## 7. Save picks

In [10]:
def save_picks():
    picks_df = pd.DataFrame(PICKS)
    out_csv = OUT_ROOT / "betsy_interactive_picks.csv"
    picks_df.to_csv(out_csv, index=False)

    with sqlite3.connect(CATALOG_DB) as conn:
        conn.execute('DROP TABLE IF EXISTS betsy_interactive_picks')
        if len(picks_df.columns) == 0:
            picks_df = pd.DataFrame(columns=["dataset", "receiver_x_m", "pick_time_s", "event_id"])
        picks_df.to_sql("betsy_interactive_picks", conn, if_exists="fail", index=False)
        conn.commit()

    print("Saved:", out_csv)
    display(picks_df)
    return picks_df

# Run after picking:
picks_df = save_picks()

Saved: /Volumes/tachyon/LBSSP_DATA/betsy_gun_alignment_v1/interactive_picker/betsy_interactive_picks.csv


,dataset,receiver_x_m,pick_time_s,event_id
